In [1]:
# pip install faiss-cpu

In [2]:
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter 

C:\Users\vigne\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
data=open('data.txt').read()

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)
chunks = splitter.split_text(data)

In [5]:
embedding_model = SentenceTransformer(
    model_name_or_path= 'sentence-transformers/all-miniLM-L6-V2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4510.00it/s]


In [6]:
embeddings = embedding_model.encode(chunks).astype('float32')

In [7]:
embeddings

array([[-0.00840593,  0.08934818,  0.00949457, ..., -0.0308043 ,
        -0.04191475,  0.03801662],
       [-0.04947576, -0.06186698,  0.00908896, ...,  0.04027702,
        -0.09283801, -0.00787669],
       [-0.02657852,  0.10831524, -0.00629124, ..., -0.05291905,
        -0.03612455, -0.00969002],
       ...,
       [ 0.02472579,  0.0337792 , -0.01883468, ...,  0.00572359,
         0.07867831,  0.05865175],
       [-0.00653822, -0.01367643, -0.01397769, ...,  0.07139353,
        -0.01209193,  0.03856924],
       [-0.01879415, -0.0268713 ,  0.02383245, ...,  0.06817986,
         0.06408651,  0.0095564 ]], shape=(1000, 384), dtype=float32)

In [8]:
dimension = embeddings.shape[1] #384

In [9]:
faiss.normalize_L2(embeddings) #Euclidean Distance

In [10]:
index = faiss.IndexFlatIP(dimension) #dot product #index local db
# index = faiss.IndexFlatL2() #euclidean distance

In [11]:
index.add(embeddings) #stored embeddings inside faiss DB

#### Search Method(R)

In [12]:
Query = 'Explain Machine Learning ?'

In [13]:
query_embeddings = embedding_model.encode(Query).astype('float32')

In [14]:
query_embeddings = query_embeddings.reshape(1,-1)
query_embeddings

array([[-1.24870632e-02, -2.93794740e-02,  3.04285679e-02,
         1.55043835e-02,  5.17589226e-02, -3.45746726e-02,
        -3.30067798e-03, -2.68255677e-02, -2.55472790e-02,
        -1.16211055e-02, -6.04345463e-02,  3.57628688e-02,
         4.54587601e-02, -8.03952366e-02, -8.29185769e-02,
        -7.45698903e-03, -3.19554433e-02,  2.52576768e-02,
        -9.97760296e-02, -1.09578453e-01,  5.82825504e-02,
        -3.63752842e-02, -6.94946945e-02,  2.50929892e-02,
         2.06710789e-02,  5.07702082e-02,  3.87980454e-02,
         6.04338907e-02,  1.48697020e-02, -9.96481162e-03,
         2.40478069e-02, -1.68943759e-02,  4.03817855e-02,
         3.85296568e-02, -8.52659494e-02,  2.62187440e-02,
        -4.54909019e-02,  4.75514829e-02,  1.19168963e-02,
         5.42534441e-02, -4.02193703e-02, -5.95136397e-02,
        -5.54297864e-03,  2.07224432e-02,  1.39104202e-01,
         8.75249207e-02, -3.82023044e-02, -6.54589906e-02,
        -1.47253489e-02, -5.12091182e-02, -1.56483650e-0

In [15]:
query_embeddings.shape

(1, 384)

In [16]:
faiss.normalize_L2(query_embeddings)

In [17]:
query_embeddings

array([[-1.24870632e-02, -2.93794740e-02,  3.04285679e-02,
         1.55043835e-02,  5.17589226e-02, -3.45746726e-02,
        -3.30067798e-03, -2.68255677e-02, -2.55472790e-02,
        -1.16211055e-02, -6.04345463e-02,  3.57628688e-02,
         4.54587601e-02, -8.03952366e-02, -8.29185769e-02,
        -7.45698903e-03, -3.19554433e-02,  2.52576768e-02,
        -9.97760296e-02, -1.09578453e-01,  5.82825504e-02,
        -3.63752842e-02, -6.94946945e-02,  2.50929892e-02,
         2.06710789e-02,  5.07702082e-02,  3.87980454e-02,
         6.04338907e-02,  1.48697020e-02, -9.96481162e-03,
         2.40478069e-02, -1.68943759e-02,  4.03817855e-02,
         3.85296568e-02, -8.52659494e-02,  2.62187440e-02,
        -4.54909019e-02,  4.75514829e-02,  1.19168963e-02,
         5.42534441e-02, -4.02193703e-02, -5.95136397e-02,
        -5.54297864e-03,  2.07224432e-02,  1.39104202e-01,
         8.75249207e-02, -3.82023044e-02, -6.54589906e-02,
        -1.47253489e-02, -5.12091182e-02, -1.56483650e-0

In [26]:
''' 
1st value - > distance
2nd value -> index
'''
distance,index_ = index.search(query_embeddings,k=5) #top 5 similar chunks
#k parameter,it decides how many features to retreive
index_

array([[831, 631, 431, 231,  31]])

In [38]:
result = ''
for i in index_[0]: #[[3488, 2208,  768, 2848,  608]]
    result+=chunks[i]+' ' #chunks[3488]
    


In [39]:
result

'Generative AI improves the consistency of training data before tokenization and vector representation. Generative AI improves the consistency of training data before tokenization and vector representation. Generative AI improves the consistency of training data before tokenization and vector representation. Generative AI improves the consistency of training data before tokenization and vector representation. Generative AI improves the consistency of training data before tokenization and vector representation. '

In [25]:
chunks,len(chunks)

(['Big data collects large datasets from distributed sources using distributed storage and parallel computation.',
  'Generative AI processes high volume information in real time using distributed storage and parallel computation.',
  'A big data pipeline stores structured and unstructured records using distributed storage and parallel computation.',
  'Text normalization normalizes noisy text before model training using distributed storage and parallel computation.',
  'Lemmatization converts words into meaningful linguistic forms using distributed storage and parallel computation.',
  'Natural language processing removes unnecessary punctuation and symbols using distributed storage and parallel computation.',
  'A data engineer handles spelling variations across documents using distributed storage and parallel computation.',
  'A language model prepares textual data for generative models using distributed storage and parallel computation.',
  'An enterprise AI system analyzes custome

In [40]:
print(index)

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000002BE332A9830> >


In [20]:
index.search

<bound method handle_Index.<locals>.replacement_search of <faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000002BE332A9830> >>